# Registration Comparison: Suite2p vs Normcorre

This notebook compares the registration quality of suite2p's built-in registration
versus the normcorre algorithm (adapted from CaImAn).

Metrics compared:
- **Correlation with template**: higher = better alignment
- **Frame-to-frame correlation**: higher = more stable
- **Residual motion**: lower = less remaining drift
- **Edge artifacts**: lower = better handling of borders
- **Visual inspection**: side-by-side comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time

import lbm_suite2p_python as lsp
from mbo_utilities import imread

# set dark background for plots
plt.style.use('dark_background')

: 

## 1. Load Data

Update the path below to point to your data file.

In [ ]:
# configure paths
DATA_PATH = Path(r"D:/data/your_file.tif")  # update this
OUTPUT_DIR = Path(r"D:/results/registration_comparison")
PLANE = 7  # which plane to compare (1-indexed)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Data: {DATA_PATH}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
# load data
arr = imread(DATA_PATH)
print(f"Shape: {arr.shape}")
print(f"Type: {type(arr).__name__}")

# extract single plane for comparison
if arr.ndim == 4:
    plane_idx = PLANE - 1
    frames = np.array(arr[:, plane_idx, :, :]).astype(np.float32)
else:
    frames = np.array(arr[:]).astype(np.float32)

n_frames, Ly, Lx = frames.shape
print(f"\nPlane {PLANE}: {n_frames} frames, {Ly} x {Lx}")

## 2. Run Suite2p Registration

In [ ]:
# run suite2p pipeline with registration only
s2p_dir = OUTPUT_DIR / "suite2p"

print("Running Suite2p registration...")
t0 = time.time()

ops_s2p = lsp.default_ops()
ops_s2p.update({
    "do_registration": 1,
    "roidetect": 0,  # skip detection for now
    "nonrigid": True,
    "block_size": [128, 128],
})

# run on the extracted frames
result_s2p = lsp.pipeline(
    DATA_PATH,
    save_path=s2p_dir,
    planes=PLANE,
    ops=ops_s2p,
    force_reg=True,
)

s2p_time = time.time() - t0
print(f"Suite2p registration: {s2p_time:.1f}s")

In [ ]:
# load suite2p registered data
s2p_ops = lsp.load_ops(s2p_dir)
s2p_reg_file = Path(s2p_ops["reg_file"])

s2p_registered = np.memmap(
    s2p_reg_file, dtype=np.int16, mode='r',
    shape=(n_frames, Ly, Lx)
).astype(np.float32)

# extract shifts
s2p_yoff = s2p_ops.get("yoff", np.zeros(n_frames))
s2p_xoff = s2p_ops.get("xoff", np.zeros(n_frames))

print(f"Suite2p shifts: y=[{s2p_yoff.min():.1f}, {s2p_yoff.max():.1f}], x=[{s2p_xoff.min():.1f}, {s2p_xoff.max():.1f}]")

## 3. Run Normcorre Registration

In [ ]:
# configure normcorre
nc_ops = lsp.normcorre_ops()
nc_ops.update({
    "max_shifts": (20, 20),
    "upsample_factor": 10,
    "pw_rigid": False,  # start with rigid
    "gSig_filt": None,
    "border_nan": False,  # use 0 instead of NaN
})

print("Running Normcorre rigid registration...")
t0 = time.time()

nc_rigid, nc_rigid_shifts, nc_template = lsp.register_frames(
    frames, ops=nc_ops
)

nc_rigid_time = time.time() - t0
print(f"Normcorre rigid: {nc_rigid_time:.1f}s")

# extract shifts
nc_rigid_yoff = np.array([s[0] for s in nc_rigid_shifts])
nc_rigid_xoff = np.array([s[1] for s in nc_rigid_shifts])
print(f"Normcorre shifts: y=[{nc_rigid_yoff.min():.1f}, {nc_rigid_yoff.max():.1f}], x=[{nc_rigid_xoff.min():.1f}, {nc_rigid_xoff.max():.1f}]")

In [ ]:
# run piecewise rigid normcorre
nc_ops_pw = lsp.normcorre_ops()
nc_ops_pw.update({
    "max_shifts": (20, 20),
    "upsample_factor": 10,
    "pw_rigid": True,
    "strides": (96, 96),
    "overlaps": (32, 32),
    "max_deviation_rigid": 3,
    "border_nan": False,
})

print("Running Normcorre piecewise-rigid registration...")
t0 = time.time()

nc_pwrigid, nc_pwrigid_shifts, _ = lsp.register_frames(
    frames, template=nc_template, ops=nc_ops_pw
)

nc_pwrigid_time = time.time() - t0
print(f"Normcorre piecewise-rigid: {nc_pwrigid_time:.1f}s")

## 4. Compute Quality Metrics

In [ ]:
def compute_registration_metrics(registered, raw=None, template=None):
    """
    Compute registration quality metrics.

    Returns dict with:
    - corr_template: mean correlation with template
    - corr_consecutive: mean correlation between consecutive frames
    - residual_motion: std of frame-to-frame differences
    - edge_nan_fraction: fraction of edge pixels that are NaN/zero
    - snr_improvement: improvement in temporal SNR vs raw (if provided)
    """
    n_frames = len(registered)

    # compute template if not provided
    if template is None:
        template = np.mean(registered, axis=0)

    # correlation with template
    template_flat = template.flatten()
    template_norm = template_flat - template_flat.mean()
    template_std = template_norm.std()

    corr_template = []
    for i in range(n_frames):
        frame_flat = registered[i].flatten()
        frame_norm = frame_flat - frame_flat.mean()
        corr = np.sum(template_norm * frame_norm) / (template_std * frame_norm.std() * len(template_flat))
        corr_template.append(corr)

    # consecutive frame correlation
    corr_consecutive = []
    for i in range(n_frames - 1):
        f1 = registered[i].flatten()
        f2 = registered[i + 1].flatten()
        f1_norm = f1 - f1.mean()
        f2_norm = f2 - f2.mean()
        corr = np.sum(f1_norm * f2_norm) / (f1_norm.std() * f2_norm.std() * len(f1))
        corr_consecutive.append(corr)

    # residual motion (frame differences)
    diffs = np.diff(registered, axis=0)
    residual_motion = np.std(diffs)

    # edge artifacts
    edge_mask = np.zeros(registered[0].shape, dtype=bool)
    edge_width = 10
    edge_mask[:edge_width, :] = True
    edge_mask[-edge_width:, :] = True
    edge_mask[:, :edge_width] = True
    edge_mask[:, -edge_width:] = True

    edge_bad = 0
    edge_total = 0
    for i in range(n_frames):
        edge_vals = registered[i][edge_mask]
        edge_bad += np.sum(np.isnan(edge_vals) | (edge_vals == 0))
        edge_total += len(edge_vals)
    edge_nan_fraction = edge_bad / edge_total if edge_total > 0 else 0

    # temporal SNR improvement
    snr_improvement = None
    if raw is not None:
        # compute temporal std / mean for a central region
        h, w = registered[0].shape
        roi = slice(h//4, 3*h//4), slice(w//4, 3*w//4)

        raw_roi = raw[:, roi[0], roi[1]]
        reg_roi = registered[:, roi[0], roi[1]]

        raw_snr = np.mean(raw_roi) / np.std(raw_roi, axis=0).mean()
        reg_snr = np.mean(reg_roi) / np.std(reg_roi, axis=0).mean()
        snr_improvement = reg_snr / raw_snr

    return {
        "corr_template_mean": np.mean(corr_template),
        "corr_template_std": np.std(corr_template),
        "corr_consecutive_mean": np.mean(corr_consecutive),
        "corr_consecutive_std": np.std(corr_consecutive),
        "residual_motion": residual_motion,
        "edge_nan_fraction": edge_nan_fraction,
        "snr_improvement": snr_improvement,
        "corr_template_ts": np.array(corr_template),
        "corr_consecutive_ts": np.array(corr_consecutive),
    }

In [ ]:
# compute metrics for each method
print("Computing metrics...")

metrics_raw = compute_registration_metrics(frames)
metrics_s2p = compute_registration_metrics(s2p_registered, raw=frames)
metrics_nc_rigid = compute_registration_metrics(nc_rigid, raw=frames, template=nc_template)
metrics_nc_pwrigid = compute_registration_metrics(nc_pwrigid, raw=frames, template=nc_template)

print("Done!")

In [ ]:
# create comparison table
import pandas as pd

comparison = pd.DataFrame({
    "Metric": [
        "Corr with template (mean)",
        "Corr with template (std)",
        "Consecutive frame corr (mean)",
        "Consecutive frame corr (std)",
        "Residual motion (lower=better)",
        "Edge artifacts (lower=better)",
        "SNR improvement",
    ],
    "Raw": [
        f"{metrics_raw['corr_template_mean']:.4f}",
        f"{metrics_raw['corr_template_std']:.4f}",
        f"{metrics_raw['corr_consecutive_mean']:.4f}",
        f"{metrics_raw['corr_consecutive_std']:.4f}",
        f"{metrics_raw['residual_motion']:.2f}",
        f"{metrics_raw['edge_nan_fraction']:.4f}",
        "1.00",
    ],
    "Suite2p": [
        f"{metrics_s2p['corr_template_mean']:.4f}",
        f"{metrics_s2p['corr_template_std']:.4f}",
        f"{metrics_s2p['corr_consecutive_mean']:.4f}",
        f"{metrics_s2p['corr_consecutive_std']:.4f}",
        f"{metrics_s2p['residual_motion']:.2f}",
        f"{metrics_s2p['edge_nan_fraction']:.4f}",
        f"{metrics_s2p['snr_improvement']:.3f}" if metrics_s2p['snr_improvement'] else "N/A",
    ],
    "Normcorre (rigid)": [
        f"{metrics_nc_rigid['corr_template_mean']:.4f}",
        f"{metrics_nc_rigid['corr_template_std']:.4f}",
        f"{metrics_nc_rigid['corr_consecutive_mean']:.4f}",
        f"{metrics_nc_rigid['corr_consecutive_std']:.4f}",
        f"{metrics_nc_rigid['residual_motion']:.2f}",
        f"{metrics_nc_rigid['edge_nan_fraction']:.4f}",
        f"{metrics_nc_rigid['snr_improvement']:.3f}" if metrics_nc_rigid['snr_improvement'] else "N/A",
    ],
    "Normcorre (pw-rigid)": [
        f"{metrics_nc_pwrigid['corr_template_mean']:.4f}",
        f"{metrics_nc_pwrigid['corr_template_std']:.4f}",
        f"{metrics_nc_pwrigid['corr_consecutive_mean']:.4f}",
        f"{metrics_nc_pwrigid['corr_consecutive_std']:.4f}",
        f"{metrics_nc_pwrigid['residual_motion']:.2f}",
        f"{metrics_nc_pwrigid['edge_nan_fraction']:.4f}",
        f"{metrics_nc_pwrigid['snr_improvement']:.3f}" if metrics_nc_pwrigid['snr_improvement'] else "N/A",
    ],
})

print("\n" + "="*80)
print("REGISTRATION QUALITY COMPARISON")
print("="*80)
print(comparison.to_string(index=False))

## 5. Visualize Results

In [ ]:
# plot correlation time series
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# template correlation
ax = axes[0]
ax.plot(metrics_raw['corr_template_ts'], alpha=0.5, label='Raw', color='gray')
ax.plot(metrics_s2p['corr_template_ts'], alpha=0.8, label='Suite2p', color='#3498db')
ax.plot(metrics_nc_rigid['corr_template_ts'], alpha=0.8, label='Normcorre (rigid)', color='#2ecc71')
ax.plot(metrics_nc_pwrigid['corr_template_ts'], alpha=0.8, label='Normcorre (pw-rigid)', color='#e74c3c')
ax.set_xlabel('Frame')
ax.set_ylabel('Correlation with Template')
ax.set_title('Template Correlation Over Time')
ax.legend(loc='lower right')
ax.set_ylim([0.8, 1.0])

# consecutive correlation
ax = axes[1]
ax.plot(metrics_raw['corr_consecutive_ts'], alpha=0.5, label='Raw', color='gray')
ax.plot(metrics_s2p['corr_consecutive_ts'], alpha=0.8, label='Suite2p', color='#3498db')
ax.plot(metrics_nc_rigid['corr_consecutive_ts'], alpha=0.8, label='Normcorre (rigid)', color='#2ecc71')
ax.plot(metrics_nc_pwrigid['corr_consecutive_ts'], alpha=0.8, label='Normcorre (pw-rigid)', color='#e74c3c')
ax.set_xlabel('Frame')
ax.set_ylabel('Correlation (frame i, i+1)')
ax.set_title('Consecutive Frame Correlation')
ax.legend(loc='lower right')
ax.set_ylim([0.9, 1.0])

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'correlation_timeseries.png', dpi=150, facecolor='black')
plt.show()

In [ ]:
# plot shift comparisons
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# suite2p shifts
ax = axes[0, 0]
ax.plot(s2p_yoff, label='Y shift', alpha=0.8)
ax.plot(s2p_xoff, label='X shift', alpha=0.8)
ax.set_xlabel('Frame')
ax.set_ylabel('Shift (pixels)')
ax.set_title('Suite2p Shifts')
ax.legend()

# normcorre rigid shifts
ax = axes[0, 1]
ax.plot(nc_rigid_yoff, label='Y shift', alpha=0.8)
ax.plot(nc_rigid_xoff, label='X shift', alpha=0.8)
ax.set_xlabel('Frame')
ax.set_ylabel('Shift (pixels)')
ax.set_title('Normcorre Rigid Shifts')
ax.legend()

# shift difference (suite2p vs normcorre)
ax = axes[1, 0]
ax.plot(s2p_yoff - nc_rigid_yoff, label='Y diff', alpha=0.8)
ax.plot(s2p_xoff - nc_rigid_xoff, label='X diff', alpha=0.8)
ax.axhline(0, color='white', linestyle='--', alpha=0.3)
ax.set_xlabel('Frame')
ax.set_ylabel('Shift Difference (pixels)')
ax.set_title('Suite2p - Normcorre Shift Difference')
ax.legend()

# histogram of shift differences
ax = axes[1, 1]
ax.hist(s2p_yoff - nc_rigid_yoff, bins=50, alpha=0.6, label='Y diff')
ax.hist(s2p_xoff - nc_rigid_xoff, bins=50, alpha=0.6, label='X diff')
ax.axvline(0, color='white', linestyle='--', alpha=0.5)
ax.set_xlabel('Shift Difference (pixels)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Shift Differences')
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shift_comparison.png', dpi=150, facecolor='black')
plt.show()

In [ ]:
# visual comparison of mean images
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

mean_raw = np.mean(frames, axis=0)
mean_s2p = np.mean(s2p_registered, axis=0)
mean_nc_rigid = np.mean(nc_rigid, axis=0)
mean_nc_pwrigid = np.mean(nc_pwrigid, axis=0)

vmin, vmax = np.percentile(mean_raw, [1, 99])

axes[0, 0].imshow(mean_raw, cmap='gray', vmin=vmin, vmax=vmax)
axes[0, 0].set_title('Raw (unregistered)')
axes[0, 0].axis('off')

axes[0, 1].imshow(mean_s2p, cmap='gray', vmin=vmin, vmax=vmax)
axes[0, 1].set_title('Suite2p')
axes[0, 1].axis('off')

axes[1, 0].imshow(mean_nc_rigid, cmap='gray', vmin=vmin, vmax=vmax)
axes[1, 0].set_title('Normcorre (rigid)')
axes[1, 0].axis('off')

axes[1, 1].imshow(mean_nc_pwrigid, cmap='gray', vmin=vmin, vmax=vmax)
axes[1, 1].set_title('Normcorre (piecewise-rigid)')
axes[1, 1].axis('off')

plt.suptitle('Mean Registered Images', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'mean_images.png', dpi=150, facecolor='black')
plt.show()

In [ ]:
# compute and display temporal std (motion blur indicator)
fig, axes = plt.subplots(2, 2, figsize=(12, 12))

std_raw = np.std(frames, axis=0)
std_s2p = np.std(s2p_registered, axis=0)
std_nc_rigid = np.std(nc_rigid, axis=0)
std_nc_pwrigid = np.std(nc_pwrigid, axis=0)

vmin_std, vmax_std = np.percentile(std_raw, [1, 99])

axes[0, 0].imshow(std_raw, cmap='hot', vmin=vmin_std, vmax=vmax_std)
axes[0, 0].set_title(f'Raw (std={std_raw.mean():.1f})')
axes[0, 0].axis('off')

axes[0, 1].imshow(std_s2p, cmap='hot', vmin=vmin_std, vmax=vmax_std)
axes[0, 1].set_title(f'Suite2p (std={std_s2p.mean():.1f})')
axes[0, 1].axis('off')

axes[1, 0].imshow(std_nc_rigid, cmap='hot', vmin=vmin_std, vmax=vmax_std)
axes[1, 0].set_title(f'Normcorre rigid (std={std_nc_rigid.mean():.1f})')
axes[1, 0].axis('off')

axes[1, 1].imshow(std_nc_pwrigid, cmap='hot', vmin=vmin_std, vmax=vmax_std)
axes[1, 1].set_title(f'Normcorre pw-rigid (std={std_nc_pwrigid.mean():.1f})')
axes[1, 1].axis('off')

plt.suptitle('Temporal Standard Deviation (lower = less motion blur)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'temporal_std.png', dpi=150, facecolor='black')
plt.show()

## 6. Summary Bar Chart

In [ ]:
# summary bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

methods = ['Raw', 'Suite2p', 'Normcorre\n(rigid)', 'Normcorre\n(pw-rigid)']
colors = ['gray', '#3498db', '#2ecc71', '#e74c3c']

# correlation with template
ax = axes[0]
vals = [
    metrics_raw['corr_template_mean'],
    metrics_s2p['corr_template_mean'],
    metrics_nc_rigid['corr_template_mean'],
    metrics_nc_pwrigid['corr_template_mean'],
]
errs = [
    metrics_raw['corr_template_std'],
    metrics_s2p['corr_template_std'],
    metrics_nc_rigid['corr_template_std'],
    metrics_nc_pwrigid['corr_template_std'],
]
ax.bar(methods, vals, color=colors, yerr=errs, capsize=5, alpha=0.8)
ax.set_ylabel('Correlation')
ax.set_title('Template Correlation\n(higher = better)')
ax.set_ylim([0.9, 1.0])

# residual motion
ax = axes[1]
vals = [
    metrics_raw['residual_motion'],
    metrics_s2p['residual_motion'],
    metrics_nc_rigid['residual_motion'],
    metrics_nc_pwrigid['residual_motion'],
]
ax.bar(methods, vals, color=colors, alpha=0.8)
ax.set_ylabel('Residual Motion (std)')
ax.set_title('Residual Motion\n(lower = better)')

# processing time
ax = axes[2]
times = [0, s2p_time, nc_rigid_time, nc_pwrigid_time]
ax.bar(methods, times, color=colors, alpha=0.8)
ax.set_ylabel('Time (seconds)')
ax.set_title('Processing Time')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'summary_comparison.png', dpi=150, facecolor='black')
plt.show()

## 7. Conclusion

In [ ]:
# determine winner
scores = {
    'Suite2p': 0,
    'Normcorre (rigid)': 0,
    'Normcorre (pw-rigid)': 0,
}

# higher template correlation = better
corrs = [
    ('Suite2p', metrics_s2p['corr_template_mean']),
    ('Normcorre (rigid)', metrics_nc_rigid['corr_template_mean']),
    ('Normcorre (pw-rigid)', metrics_nc_pwrigid['corr_template_mean']),
]
winner = max(corrs, key=lambda x: x[1])[0]
scores[winner] += 1

# lower residual motion = better
residuals = [
    ('Suite2p', metrics_s2p['residual_motion']),
    ('Normcorre (rigid)', metrics_nc_rigid['residual_motion']),
    ('Normcorre (pw-rigid)', metrics_nc_pwrigid['residual_motion']),
]
winner = min(residuals, key=lambda x: x[1])[0]
scores[winner] += 1

# higher consecutive correlation = better
consec = [
    ('Suite2p', metrics_s2p['corr_consecutive_mean']),
    ('Normcorre (rigid)', metrics_nc_rigid['corr_consecutive_mean']),
    ('Normcorre (pw-rigid)', metrics_nc_pwrigid['corr_consecutive_mean']),
]
winner = max(consec, key=lambda x: x[1])[0]
scores[winner] += 1

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"\nScores (wins across metrics):")
for method, score in sorted(scores.items(), key=lambda x: -x[1]):
    print(f"  {method}: {score}/3")

best = max(scores.items(), key=lambda x: x[1])[0]
print(f"\nRecommended: {best}")

print(f"\nProcessing times:")
print(f"  Suite2p: {s2p_time:.1f}s")
print(f"  Normcorre (rigid): {nc_rigid_time:.1f}s")
print(f"  Normcorre (pw-rigid): {nc_pwrigid_time:.1f}s")

In [ ]:
# save results
results = {
    'metrics_raw': {k: v for k, v in metrics_raw.items() if not k.endswith('_ts')},
    'metrics_s2p': {k: v for k, v in metrics_s2p.items() if not k.endswith('_ts')},
    'metrics_nc_rigid': {k: v for k, v in metrics_nc_rigid.items() if not k.endswith('_ts')},
    'metrics_nc_pwrigid': {k: v for k, v in metrics_nc_pwrigid.items() if not k.endswith('_ts')},
    'times': {
        'suite2p': s2p_time,
        'normcorre_rigid': nc_rigid_time,
        'normcorre_pwrigid': nc_pwrigid_time,
    },
    'scores': scores,
}

np.save(OUTPUT_DIR / 'comparison_results.npy', results)
print(f"\nResults saved to: {OUTPUT_DIR / 'comparison_results.npy'}")